# Incompressible LES: Pitz Daily with a scikit-learn Closure

## Preamble

In [ ]:
# Standard Library
from pathlib import Path
import foamnordic as fno
import numpy as np
import onsaemiro as osm

# Machine Learning
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearnex import patch_sklearn
patch_sklearn()

### Directories

In [ ]:
# Case Type and Name
CASE_TYPE = "les"
CASE_NAME = "pitzDailySmag"
CATEGORY = "incompressible"

# FoamNordic Project Directory
PROJECT_DIR = Path.cwd().resolve()
BASE_DIR = next((path for path in (PROJECT_DIR, *PROJECT_DIR.parents) if (path / "tutorials" / "openfoam_tutorials").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the FoamNordic source tree")
MAIN_DIR = BASE_DIR / "tutorials" / "foamnordic_tutorials" / CATEGORY
OF_SCRIPT_DIR = BASE_DIR / "tutorials" / "openfoam_tutorials" / CATEGORY / CASE_TYPE / CASE_NAME

# Output Directory
MODEL_DIR = MAIN_DIR / "model"
OUTPUT_DIR = MAIN_DIR / "output"

for directory in [MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

### Configuration

In [ ]:
# DataGraph Configuration
FIG_X, FIG_Y = 3.5, 2.55
FIGURE_SIZE = (FIG_X, FIG_Y)
PALETTE = osm.get_palette("OKABE_ITO")
osm.set_style(figure_size=FIGURE_SIZE)

In [ ]:
# HPC configuration
ACCOUNT = "<allocation-account>" # e.g. project_xxxxxxx
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

model_scheduler = fno.Slurm.model(
    cpus_per_task=1,
    mem_per_cpu=MEM_PER_CPU
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=model_scheduler
)

In [ ]:
# FoamNordic configuration
SEED = 42


## Example - Closure Modelling

### Synthetic Data Generation

In [ ]:
# Smagorinsky model coefficients
C_K = 0.0265463553
C_E = 1.048

# Training configuration
N_TRAIN = 30000
N_NEIGHBORS = 5
N_ESTIMATORS = 100

In [ ]:
# Smagorinsky model for LES
def smagorinsky_nut(velocity_grad, delta):
    symm_grad = 0.5 * (velocity_grad + np.swapaxes(velocity_grad, -1, -2))

    trace_grad = np.trace(symm_grad, axis1=-2, axis2=-1)

    identity = np.eye(3, dtype=velocity_grad.dtype)

    dev_symm_grad = (
        symm_grad
        - (1.0 / 3.0)
        * trace_grad[..., None, None]
        * identity
    )

    dev_contraction = np.sum(
        dev_symm_grad * symm_grad,
        axis=(-2, -1),
    )

    coeff_a = C_E / delta
    coeff_b = (2.0 / 3.0) * trace_grad
    coeff_c = (2.0 * C_K * delta * dev_contraction)

    discriminant = (coeff_b**2 + 4.0 * coeff_a * coeff_c)

    sqrt_k = (
        -coeff_b
        + np.sqrt(np.maximum(discriminant, 0.0))
    ) / (
        2.0 * coeff_a
    )

    sqrt_k = np.maximum(sqrt_k, 0.0)

    nut = C_K * delta * sqrt_k

    return nut

In [ ]:
# Load vanilla OpenFOAM results
post = fno.Postprocess.Case(source=OF_SCRIPT_DIR)
times = tuple(time for time in post.times if time > 0.0)

TRAIN_TIMES = times[:-1]
TEST_TIME = times[-1]

In [ ]:
# Load cell volume and calculate LES filter width
cell_volume = post.field(name="V", time_idx=None, physical_time=TEST_TIME).reshape(-1)
delta = np.cbrt(cell_volume)

# Load training trajectories
train_grad = []

for physical_time in TRAIN_TIMES:
    velocity_grad = post.field(name="gradU", time_idx=None, physical_time=physical_time).reshape(-1, 3, 3)
    train_grad.append(velocity_grad)

train_grad = np.concatenate(train_grad)
train_delta = np.tile(delta, len(TRAIN_TIMES))

# Load the held-out final trajectory
test_grad = post.field(name="gradU", time_idx=None, physical_time=TEST_TIME).reshape(-1, 3, 3)
test_delta = delta.copy()

print(f"Training times: {TRAIN_TIMES}")
print(f"Test time: {TEST_TIME}")
print(f"Training cells: {len(train_grad)}")
print(f"Test cells: {len(test_grad)}")

In [ ]:
# Calculate the mathematical Smagorinsky targets
y_train_full = smagorinsky_nut(velocity_grad=train_grad, delta=train_delta)
y_test = smagorinsky_nut(velocity_grad=test_grad, delta=test_delta)

# Pack grad(U) and delta into the FoamNordic model contract
X_train_full = np.column_stack([train_grad.reshape(len(train_grad), -1), train_delta])
X_test = np.column_stack([test_grad.reshape(len(test_grad), -1), test_delta])

# Select a stratified training subset
if len(X_train_full) > N_TRAIN:
    nut_bins = np.unique(np.quantile(y_train_full, np.linspace(0.0, 1.0, 11)))
    labels = np.digitize(y_train_full, nut_bins[1:-1])

    X_train, _, y_train, _ = train_test_split(
        X_train_full,
        y_train_full,
        train_size=N_TRAIN,
        random_state=SEED,
        stratify=labels,
    )
else:
    X_train = X_train_full
    y_train = y_train_full

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Feature Scaling
scaler_X = StandardScaler().fit(X_train)

inactive_features = scaler_X.var_ < 1.0e-20

scaler_X.mean_[inactive_features] = 0.0
scaler_X.scale_[inactive_features] = 1.0
scaler_X.var_[inactive_features] = 1.0

X_train_scaled = scaler_X.transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

### ML Modelling

In [ ]:
# Voting Regressor with Extra Trees and K-Nearest Neighbors
N_NEIGHBORS = 5

# Extra Trees Regressor
smagorinsky_regressor = ExtraTreesRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)

# Fit the Voting Regressor
smagorinsky_regressor.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = smagorinsky_regressor.predict(X_test_scaled)

In [ ]:
# Compute metrics
metric_r2 = r2_score(y_test, y_pred)
metric_mse = mean_squared_error(y_test, y_pred)
metric_mae = mean_absolute_error(y_test, y_pred)
metric_relative_error = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
metric_99_percentile_error = np.percentile(np.abs((y_test - y_pred) / y_test), 99) * 100

table = osm.TableMaker(
    columns=["R2", "MSE", "MAE", "Relative Error (%)", "99th Percentile Error (%)"],
    title="Smagorinsky Model Performance Metrics",
    mode="static"
)

table.add_row([
    f"{metric_r2:.6f}",
    f"{metric_mse:.6e}",
    f"{metric_mae:.6e}",
    f"{metric_relative_error:.6f}",
    f"{metric_99_percentile_error:.6f}"
])
table.display()

In [ ]:
# Model Export
# Define the FoamNordic model path
MODEL_PATH = MODEL_DIR / "smagorinsky_skl.fnom"

# Export the trained model in FoamNordic format
fno.Export.sklearn(
    model=smagorinsky_regressor,
    path=MODEL_PATH,
    inputs={
        "velocity_grad": fno.Tensor.tensor(components=9, dtype="float64"),
        "delta": fno.Tensor.scalar(dtype="float64"),
    },
    outputs={
        "nut": fno.Tensor.scalar(dtype="float64"),
    },
    name="smagorinsky",
    x_scaler=scaler_X,
    y_scaler=None,
    backend="auto",
    runtime="sklearn",
    input_format="numpy",
    verbose=True,
)

### Smagorinsky Closure

In [ ]:
# Define the Smagorinsky closure
smagorinsky_closure = fno.Closure(
    name="nutFjord",
    operator=fno.Operator.model(path=MODEL_PATH),
    inputs={
        "velocity_grad": fno.Field.grad(value="U"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "nut": fno.Field(name="nut"),
    },
    key=None,
)

### Case Definition

In [ ]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
    ranks=N_TASKS,
    integration=None,
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True)

### Submit Job

In [ ]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(
    case=case,
    closures=(smagorinsky_closure,),
    transforms=(),
    observations=(),
    placement=fno.Attached(model_cpus_per_node=None, data_path="auto"),
    scheduler=scheduler,
    name=None,
    verbose=True,
    numerics="convenient",
)

In [ ]:
# Launch the workload
run = longship.launch(
    readiness_timeout=120.0,
    termination_grace=30.0,
    orphan_timeout=30.0,
    start_timeout=900.0,
    verbose=None,
    profile_every=0,
    receive_memory_bytes=None,
)

In [ ]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [ ]:
# Summary of the job result
result.summary(style="compact", display=True)

### Post-processing

In [ ]:
# Postprocessing
post = result.postprocess

velocity = post.field(name="U", time_idx=-1, physical_time=None)
pressure = post.field(name="p", time_idx=-1, physical_time=None)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    fields=["U", "p", "nut"],
    time_idx=-1,
    physical_time=None,
    verbose=True,
)